In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent)) 

In [2]:
fs = 12000  # how many numbers per second are in the recording
window_size = 2048  # size of each bite-sized chunk
overlap = 0.5
step = int(window_size * (1 - overlap))

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

In [9]:
def get_label(filename):
    name = filename.stem
    if "Normal" in name: return "Normal"
    elif "_IR_" in name: return "InnerRace"
    elif "_B_" in name: return "Ball"
    elif "OR@6" in name: return "OuterRace"
    return None

def window_signal(signal, window_size, step):
    windows = []
    for start in range(0, len(signal) - window_size, step):
        windows.append(signal[start:start+window_size])
    return windows

data_dir = Path(r"C:\Users\user\motor_pump_predictive_system\data\raw\CWRU_Bearing_NumPy\Data")
files = list(data_dir.glob("**/*_7_DE12.npz")) + list(data_dir.glob("**/*_Normal.npz"))
print(len(files))

rows = []
for f in files:
    label = get_label(f)
    if label is None: continue
    d = np.load(f)
    signal = d['DE']
    if np.std(signal) < 1e-6: continue  # skip broken/flat recordings

    for w in window_signal(signal, window_size, step):
        if np.std(w) < 1e-6: continue  # skip dead/flat chunks
        rows.append({"window": w, "label": label, "source_file": f.name})

df = pd.DataFrame(rows)
print(df['label'].value_counts())
df.to_pickle(
    r"C:\Users\user\motor_pump_predictive_system\data\processed\windows.pkl")

24
label
Normal       1653
InnerRace     472
OuterRace     471
Ball          469
Name: count, dtype: int64


In [10]:
print(len(files))


24


In [11]:
import numpy as np
from scipy.stats import kurtosis, skew
from scipy.fft import rfft, rfftfreq

In [12]:
FS = 12000

def extract_features(window):
    feats = {}
    feats['mean'] = np.mean(window)
    feats['std'] = np.std(window)
    feats['rms'] = np.sqrt(np.mean(window**2))
    feats['skew'] = skew(window)
    feats['kurtosis'] = kurtosis(window)
    feats['ptp'] = np.ptp(window)
    feats['crest_factor'] = np.max(np.abs(window)) / (feats['rms'] + 1e-9)

    freqs = rfftfreq(len(window), 1/FS)
    fft_vals = np.abs(rfft(window))
    feats['dominant_freq'] = freqs[np.argmax(fft_vals)]
    feats['spectral_energy'] = np.sum(fft_vals**2)
    feats['spectral_entropy'] = -np.sum(
        (fft_vals/np.sum(fft_vals)) * np.log(fft_vals/np.sum(fft_vals) + 1e-12)
    )
    return feats
